---

## 🚀 <u>Primer Avance Proyecto Integrador</u>

- Si bien la consigna incluye el uso de AWS Lake Formation, se decidió no activarlo en esta implementación debido a restricciones de costos, manteniendo su consideración como mejora futura en un entorno productivo.

# Se creó un bucket S3 con las siguientes configuraciones:

Región: us-east-2

Versionado: habilitado

Encriptación: SSE-S3

Bloqueo de acceso público

Control de acceso mediante políticas IAM

# - El Data Lake adopta una arquitectura por capas:

Se crearon 3 buckets:

pi4.bronze/ → Datos crudos

pi4.silver/ → Datos limpios

pi4.gold/ → Datos analíticos

# - Diagrama detallado de la arquitectura del pipeline

![Arquitectura](../Img/arq.png)

# - Definición explícita del propósito de cada capa del Data Lake

🟤 Bronze 

Es la capa de entrada.

Guarda los datos tal como llegan desde las fuentes (API, PostgreSQL, streaming, etc.).

No se aplican transformaciones.

Permite trazabilidad y reprocesamiento si cambian las reglas.

⚪ Silver 

Es la capa de procesamiento.

Se limpian y estructuran los datos.

Se eliminan duplicados.

Se normalizan tipos de datos.

Se aplican reglas de calidad.

Se particionan para mejor rendimiento.

🟡 Gold 

Es la capa analítica.

Se generan métricas y KPIs.

Se crean agregaciones por región u otras dimensiones.

Se preparan tablas listas para BI.

# - Selección y Justificación del Stack Tecnológico

🔹 Almacenamiento – Amazon S3

Justificación:

Alta durabilidad 

Escalabilidad prácticamente ilimitada

Bajo costo en Free Tier

Integración nativa con servicios AWS

Adecuado para implementar un Data Lake por su naturaleza orientada a objetos.

🔹 Procesamiento – Apache Spark

Justificación:

Procesamiento distribuido

Escalable horizontalmente

Compatible con procesamiento batch y streaming

Ideal para arquitectura por capas

Permite manejar grandes volúmenes de datos de manera eficiente.

🔹 Orquestación – Apache Airflow

Justificación:

Gestión de dependencias entre tareas

Programación automatizada

Reintentos y monitoreo

Separación clara entre lógica y ejecución

Es adecuado para coordinar pipelines ETLT complejos.

🔹 Streaming – Apache Kafka

Justificación:

Alta tolerancia a fallos

Procesamiento en tiempo real

Escalabilidad

Integración con sistemas modernos

Permite manejar datos incrementales en tiempo real.

🔹 Base de Datos – PostgreSQL

Justificación:

Motor relacional robusto

Integración sencilla con herramientas Python

Fuente estructurada confiable

🔹 Gobernanza – IAM Policies

Si bien se evaluó AWS Lake Formation, no se implementó por restricciones de costo y alcance.

La gobernanza se realiza mediante:

Políticas IAM

Control de acceso en S3

Encriptación server-side

# - Identificación y Análisis de las Fuentes de Datos

Se nos dieron las siguientes preguntas de negocio:

1) ¿Cómo varía el potencial solar estimado a lo largo del día y del mes en Riohacha y en la Patagonia?

2) ¿Qué patrones históricos se observan en el potencial eólico de ambas ubicaciones?

3) ¿Qué condiciones climáticas están asociadas con reducciones significativas en el potencial renovable?

4) ¿Cómo se comportan las predicciones meteorológicas en comparación con las condiciones observadas en el pasado reciente?

5) ¿Cuáles fueron los días con mayor y menor potencial energético en cada ubicación durante el periodo de análisis?

El Data Lake fue diseñado para responder preguntas sobre el comportamiento del potencial solar y eólico en Riohacha y la Patagonia.

Para ello se utilizan datos meteorológicos históricos y pronosticados, incluyendo radiación solar, velocidad del viento, nubosidad y temperatura.

En la capa Bronze se almacenan los datos crudos para garantizar trazabilidad.
En Silver se limpian, estructuran y normalizan las series temporales.
En Gold se generan agregaciones diarias y mensuales, métricas comparativas y rankings que permiten:

Analizar variaciones horarias y estacionales.

Detectar patrones históricos.

Identificar condiciones climáticas que reducen el potencial.

Comparar predicciones con datos observados.

Determinar los días de mayor y menor generación estimada.




--- 

## 🚀 <u>Segundo Avance Proyecto Integrador</u>

# 1) Configuración de Airbyte Cloud (Free Tier)

Se utilizó Airbyte en su versión Cloud (Free Tier) como herramienta de ingesta dentro del pipeline ELT.

El objetivo fue extraer datos desde:

Una API pública (OpenWeather)

Los datos se almacenan en la capa raw/ dentro de un bucket (pi4.bronce) de Amazon S3.

# 2) Configuración del conector API

 Se optó por una configuración desacoplada creando:
 
 ✔ 2 Sources

- Openweather (para Patagonia)

- OpenweatherRH (para Rio Hacha)

✔ 2 Destinations

- s3 (para Patagonia)

- s3rh (para Rio Hacha)

✔ 2 Connections

- Openweather → S3 (para Patagonia)

- OpenweatherRH → S3rh (para Rio Hacha)

# Frecuencia de sincronización

- Debido a las limitaciones del Free Tier se ejecuta una vez por día.

# Validación de datos

Se verificó:

✔ Presencia de archivos en S3
✔ Formato JSON comprimido
✔ Correcta separación por ubicación
✔ Estructura coherente con la API


--- 

## 🚀 <u>Tercer avance Proyecto Integrador</u>

- Se desarrolló un script en PySpark para procesar datos meteorológicos provenientes de múltiples fuentes dentro del Data Lake. El pipeline integra datos históricos en formato JSON y datos de streaming en formato JSONL comprimido (gzip) almacenados en la capa Bronze.

- Mediante Apache Spark se realizó la unificación de esquemas, normalización de campos y transformación a un formato estructurado optimizado (Parquet), generando un dataset consolidado en la capa Silver. El procesamiento se ejecutó de manera distribuida, aprovechando el paralelismo del cluster y generando archivos particionados junto con el indicador de ejecución exitosa (_SUCCESS).

- Este proceso permite disponer de datos limpios, estructurados y listos para análisis posteriores orientados a responder las preguntas de negocio definidas.


--- 

## 🚀 <u>Tercer avance Proyecto Integrador</u>

La arquitectura del pipeline utiliza:

## Contexto del Pipeline de Datos

En esta etapa del proyecto se desarrolló un proceso de transformación de datos utilizando Apache Spark con el objetivo de preparar los datos almacenados en el Data Lake para su posterior análisis.
La arquitectura implementada sigue un enfoque de **Data Lake por capas**, donde los datos pasan por distintos niveles de procesamiento:

Bronze → Silver → Gold

El almacenamiento del Data Lake se encuentra en Amazon S3, mientras que el procesamiento distribuido es ejecutado mediante Spark desplegado en una instancia Amazon EC2.

La ejecución del proceso ETL es orquestada mediante Apache Airflow, que dispara el script de PySpark dentro del contenedor del clúster de Spark.

---

# Objetivo del Script ETL
El script desarrollado tiene como objetivo principal:

1. Leer datos meteorológicos desde la capa Bronze del Data Lake.

2. Normalizar las estructuras de los datasets provenientes de distintas fuentes.

3. Integrar datos históricos y datos de ingestión continua.

4. Detectar automáticamente la ubicación geográfica de los registros.

5. Generar un dataset limpio y consistente para análisis.

6. Almacenar los resultados en la capa Silver en formato Parquet particionado.

---

# Descripción del Proceso de Transformación

El script ETL implementa una serie de etapas de procesamiento utilizando PySpark.
En primer lugar se crea una sesión de Spark, que representa el punto de entrada al motor de procesamiento distribuido.
Durante esta etapa se configuran parámetros importantes como:

- el nombre de la aplicación Spark
- el proveedor de credenciales para acceder a Amazon S3
- el uso de compresión Snappy para archivos Parquet

---

## Lectura de datos desde la capa Bronze

Una vez inicializada la sesión de Spark, el script procede a cargar los datasets almacenados en Amazon S3.
Se leen dos conjuntos de datos: **Datos históricos** y **Datos de streaming**. Ambos datasets son cargados en memoria como **DataFrames de Spark**.

---

## Normalización de la estructura de datos

Los datasets provenientes de ambas fuentes presentan estructuras JSON diferentes. esta etapa permite obtener dos datasets con **la misma estructura de columnas**, lo cual es necesario para poder integrarlos posteriormente.

Se realiza una normalización que incluye:

- selección de columnas relevantes
- extracción de campos anidados dentro del JSON
- conversión de tipos de datos
- renombrado de columnas

Las variables seleccionadas para el análisis incluyen:

- timestamp meteorológico
- latitud
- longitud
- temperatura
- humedad

---

## Detección automática de la ubicación

La ubicación geográfica se detecta automáticamente. 
Para los datos históricos, la ubicación se obtiene desde el campo: "city_name".

## Integración de datasets

Una vez que los datasets han sido normalizados, se procede a combinarlos mediante una operación de unión, permitiendo integrar datos provenientes de distintas fuentes dentro de un único dataset analítico.
El resultado es un conjunto de datos unificado que contiene:

- datos históricos
- datos de ingestión continua
- múltiples ubicaciones geográficas

---

## Escritura de datos en la capa Silver

Finalmente, el dataset resultante se almacena en la capa Silver del Data Lake.
Para esta etapa se utiliza el formato **Parquet**.

El almacenamiento se realiza con particionamiento, lo cual optimiza el acceso a los datos durante consultas analíticas.
El esquema de particionamiento utilizado es: **location / year / month**

Esto permite que las consultas puedan acceder únicamente a las particiones relevantes, reduciendo el volumen de datos que debe procesarse.

la ruta resultante es: **s3a://pi4.silver/weather_normalized/**.


---

# Conclusión

Este proceso de desarrollado permite transformar datos meteorológicos provenientes de distintas fuentes en un dataset unificado y optimizado para análisis.

La utilización de Apache Spark permite realizar transformaciones distribuidas de manera eficiente, mientras que Amazon S3 proporciona un almacenamiento escalable para el Data Lake.

La orquestación mediante Apache Airflow permite ejecutar el pipeline de forma controlada y repetible dentro de la arquitectura del proyecto.

El uso de particionamiento por ubicación y dimensiones temporales mejora significativamente el rendimiento de consultas analíticas y facilita la escalabilidad del sistema a nuevas regiones geográficas.

In [ ]:
# ==============================
# LECTURA DE DATOS DESDE BRONZE
# ==============================

print(">>> Leyendo datos históricos desde Bronze")

df_hist = spark.read \
    .option("multiline", "true") \
    .json("s3a://pi4.bronce/historicos/*")

print(">>> Leyendo datos de streaming")

df_stream = spark.read.json(
    "s3a://pi4.bronce/stream/*/*/*"
)

# ==============================
# NORMALIZACIÓN HISTÓRICOS
# ==============================

df_hist_norm = df_hist.select(
    col("dt").cast("long"),
    col("lat").cast("double"),
    col("lon").cast("double"),
    col("main.temp").cast("double").alias("temp"),
    col("main.humidity").cast("integer").alias("humidity"),
    col("city_name")
)

df_hist_norm = df_hist_norm.withColumn(
    "location",
    lower(col("city_name"))
).drop("city_name")


# ==============================
# NORMALIZACIÓN STREAMING
# ==============================

df_stream = df_stream.withColumn(
    "file_path",
    input_file_name()
)

df_stream = df_stream.withColumn(
    "location",
    regexp_extract("file_path", "stream/(.*?)/", 1)
)

df_stream_norm = df_stream.select(
    col("_airbyte_data.current.dt").cast("long").alias("dt"),
    col("_airbyte_data.lat").cast("double").alias("lat"),
    col("_airbyte_data.lon").cast("double").alias("lon"),
    col("_airbyte_data.current.temp").cast("double").alias("temp"),
    col("_airbyte_data.current.humidity").cast("integer").alias("humidity"),
    col("location")
)


# ==============================
# UNIÓN DE DATASETS
# ==============================

df_final = df_hist_norm.unionByName(df_stream_norm)


# ==============================
# CREACIÓN DE PARTICIONES
# ==============================

df_final = df_final.withColumn(
    "date",
    from_unixtime("dt")
)

df_final = df_final.withColumn(
    "year",
    year("date")
).withColumn(
    "month",
    month("date")
)


# ==============================
# ESCRITURA EN SILVER
# ==============================

df_final.write \
    .mode("overwrite") \
    .partitionBy("location","year","month") \
    .parquet("s3a://pi4.silver/weather_normalized/")


print(">>> ETL FINALIZADO")

spark.stop()